## Creating MCP- Langchain agent for accessing MongoDB 

In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning)

from langchain.chat_models import init_chat_model

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")

#llm = init_chat_model(model="qwen/qwen3-32b", model_provider="Groq")
llm_primary = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


/Users/nali/Documents/YTLLMs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MONGODB_URI = os.getenv("MONGODB_URI")

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

## Connect your client with the MongoDB-MCP-server 

In [4]:
github_access_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

In [5]:
client = MultiServerMCPClient({
"mongodb":{
    "transport":"stdio",
    "command":"npx",
    "args":[
        "-y",
        "mongodb-mcp-server@latest",
        "--loggers",
        "stderr"
    ],
    "env":{
        "MDB_MCP_CONNECTION_STRING":MONGODB_URI
    }
},

"github":{
    "transport":"stdio",
    "command":os.path.expanduser('~/bin/github-mcp-server'),
    'args':['stdio'],
    "env":{
        "GITHUB_PERSONAL_ACCESS_TOKEN":github_access_token
    } 

}

})

In [6]:
tools = await client.get_tools()

In [7]:
for tool in tools: 
    print(tool.name)

aggregate-db
aggregate
collection-indexes
collection-schema
collection-storage-size
connect
count
create-collection
create-index
db-stats
delete-many
disconnect
drop-collection
drop-database
drop-index
explain
export
find
insert-many
list-collections
list-connections
list-databases
mongodb-logs
rename-collection
update-many
list-knowledge-sources
search-knowledge
add_comment_to_pending_review
add_issue_comment
add_reply_to_pull_request_comment
assign_copilot_to_issue
create_branch
create_or_update_file
create_pull_request
create_repository
delete_file
fork_repository
get_commit
get_file_contents
get_label
get_latest_release
get_me
get_release_by_tag
get_tag
get_team_members
get_teams
issue_read
issue_write
list_branches
list_commits
list_issue_fields
list_issue_types
list_issues
list_pull_requests
list_releases
list_repository_collaborators
list_tags
merge_pull_request
pull_request_read
pull_request_review_write
push_files
request_copilot_review
search_code
search_commits
search_issues

## Create Deepagent agent with mcp_tools

In [8]:
from deepagents import create_deep_agent

prompt="""You are a helpful mongodb assistant. 
use mogodb tools for connecting and acceesing mongodb database and answer based on user query.
use github tools for accessing github information.
"""

deepagent= create_deep_agent(
    model=  gemma,
    tools=tools,
    system_prompt=prompt
)

## Test the agent

In [9]:
user_query= """How many documents are there in the collection?
 database: University, collection: students
 """

In [17]:
user_query = """Read the README.md file from the github repo and give me 5 links about langchain documentation
 File link: https://github.com/nawazali20-code/MCP-Agent-Langchain/blob/main/README.md
"""

In [20]:
user_query = """Read the README.md file from the github repo and give me 5 links about langchain documentation
 File link: https://github.com/nawazali20-code/MCP-Agent-Langchain/blob/main/README.md

 create a new collection in MongoDB name as Links with attributes title and link and then insert those 5 extracted links and titles. 
 Database: University  
 """

In [21]:
from langchain.messages import SystemMessage,HumanMessage

try:
    result = await deepagent.ainvoke({
        "messages":[
            SystemMessage(content="You a helpful assistant."),
            HumanMessage(content=user_query)
        ]
    })
except Exception as e:
    print(f"Error happened during invoke. {e}")

In [22]:
print(result["messages"][-1].content)

Based on the content of the `README.md` file from the provided repository, here are 5 links related to Langchain documentation:

1.  **RAG:** https://docs.langchain.com/oss/python/deepagents/rag
2.  **Langgraph workflows:** https://docs.langchain.com/oss/python/langgraph/workflows-agents
3.  **Langgraph memory:** https://docs.langchain.com/oss/python/langgraph/add-memory#manage-checkpoints
4.  **Langgraph docs:** https://docs.langchain.com/oss/python/langgraph/overview
5.  **Langchain's middleware overview:** https://docs.langchain.com/oss/python/langchain/middleware/overview

I have successfully:
1.  Created a new collection named `Links` in the `University` database.
2.  Inserted the 5 extracted links and their titles into this `Links` collection.


In [12]:
user_query= """Show me the document information with field information:
name:James Cercone
for database: University, collection: students
"""

In [13]:
user_query= """Modify the following record:
current name:James Cercone modify to name: James Chase 
for database: University, collection: students
"""

In [14]:
user_query= """Add the following new document:
name:Ethan Nawaz 
dept_name:Computer Science
gpa:3.99
credit_hours:112 
for database: University, collection: students
"""

In [15]:
user_query= """Delete the following document:
name:Ethan Nawaz  
for database: University, collection: students
"""